# BigQuery Dataset Insights (Knowledge Catalog DataScans) 자동화

이 노트북은 BigQuery 데이터셋 레벨에 대해 Gemini 기반의 설명 작성을 지원하는 'DATA_DOCUMENTATION' 타입의 Knowledge Catalog DataScan 리소스를 생성하고 실행하여, 데이터셋 설명을 자동으로 생성하고 동기화하는 파이프라인입니다.

In [ ]:
import os
import json
import time
import subprocess
import urllib.request
import urllib.error
import ssl
from google.cloud import bigquery

# mTLS 오류 방지를 위해 클라이언트 인증서 사용 비활성화 설정 (Mac 환경 등에서 필요)
os.environ["GOOGLE_API_USE_CLIENT_CERTIFICATE"] = "false"

# 1. GCP 프로젝트 및 리전 정보 정의
# gcloud CLI를 사용하여 현재 활성화된 Google Cloud 프로젝트 ID를 자동으로 설정합니다.
project_proc = subprocess.run(
    ["gcloud", "config", "get-value", "project"],
    capture_output=True, text=True, check=True
)
project_lines = [line.strip() for line in project_proc.stdout.split("\n") if line.strip() and not line.strip().startswith("WARNING:")]
PROJECT_ID = project_lines[-1]
DATASET_ID = "thelook_ecommerce"
LOCATION = "asia-northeast3"  # Knowledge Catalog DataScan 실행 리전 (서울 리전)

# 2. REST API 호출용 인증 토큰 발급
# Google Cloud API 호출을 위한 OAuth2 임시 Access Token을 가져와 헤더 인증에 사용합니다.
token_proc = subprocess.run(
    ["gcloud", "auth", "print-access-token"],
    capture_output=True, text=True, check=True
)
token_lines = [line.strip() for line in token_proc.stdout.split("\n") if line.strip() and not line.strip().startswith("WARNING:")]
ACCESS_TOKEN = token_lines[-1]

# API 인증서 통과를 위한 SSL 컨텍스트 우회 설정 (로컬 환경 개발 및 검증용)
ssl_context = ssl._create_unverified_context()

# BigQuery API 호출용 클라이언트 초기화
bq_client = bigquery.Client(project=PROJECT_ID)

print(f"Project: {PROJECT_ID}, Dataset: {DATASET_ID}")
print("인프라 및 인증 설정 완료!")

In [ ]:
def make_rest_request(url, method="GET", body_dict=None):
    """
    OAuth2 Bearer 토큰 인증 헤더를 동적으로 생성하여 Google Cloud Dataplex REST API를 호출합니다.
    """
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }
    
    data = json.dumps(body_dict).encode("utf-8") if body_dict else None
    req = urllib.request.Request(url, headers=headers, method=method)
    
    try:
        with urllib.request.urlopen(req, data=data, context=ssl_context) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        err_msg = e.read().decode("utf-8")
        raise Exception(f"HTTP {e.code} - {err_msg}")

In [ ]:
def get_or_create_dataset_datascan(dataset_id):
    """
    대상 데이터셋에 대해 Gemini 기반의 설명 작성을 지원하는 'DATA_DOCUMENTATION' 타입의 Knowledge Catalog DataScan 리소스를 조회하고, 없으면 새로 생성합니다.
    """
    scan_id = f"ds-thelook-ds-{dataset_id}".lower().replace("_", "-")
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    
    try:
        scan = make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Dataset DataScan 리소스 로드 성공: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e):
            print(f"  -> 새로운 Dataset DataScan 생성 중: {scan_id}...")
            create_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}"
            body = {
                "data": {
                    "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}"
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_DOCUMENTATION",
                "dataDocumentationSpec": {
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            
            # 리소스 생성이 완료될 때까지 LRO 상태 폴링 대기
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    if "error" in op_status:
                        raise Exception(f"Dataset DataScan 생성 실패: {op_status['error']}")
                    break
                time.sleep(2)
            print(f"  -> Dataset DataScan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

def run_datascan_and_wait(scan_id):
    """
    Knowledge Catalog DataScan의 실행 Job을 기동하고, 완료될 때까지 주기적으로 상태를 조회(폴링)하여 대기합니다.
    """
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    print(f"  -> DataScan 실행 요청 중...")
    run_res = make_rest_request(run_url, method="POST")
    
    job_name = run_res["job"]["name"]
    job_id = run_res["job"]["uid"]
    print(f"  -> 실행 Job ID: {job_id} (완료 대기 시작...)")
    
    job_url = f"https://dataplex.googleapis.com/v1/{job_name}"
    while True:
        job = make_rest_request(job_url, method="GET")
        state = job.get("state")
        print(f"     [폴링] 현재 상태: {state}")
        
        if state == "SUCCEEDED":
            print("  -> 설명 생성 성공!")
            break
        elif state in ["FAILED", "CANCELLED"]:
            raise Exception(f"DataScan Job 오류 발생: {state}")
            
        time.sleep(10)

In [ ]:
def fetch_dataset_generated_description(dataset_id):
    """
    Knowledge Catalog Entry API를 호출하여 생성된 데이터셋 설명(Descriptions)의 세부 속성(Aspect) 데이터 본문을 조회합니다.
    """
    entry_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}?view=ALL"
    entry_data = make_rest_request(entry_url, method="GET")
    
    aspects = entry_data.get("aspects", {})
    desc_key = [k for k in aspects.keys() if "descriptions" in k]
    if not desc_key:
        print(f"  -> [{dataset_id}] 생성된 설명(Descriptions) Aspect가 존재하지 않습니다.")
        return None
        
    return aspects[desc_key[0]]["data"]

In [ ]:
def apply_and_publish_dataset_description(dataset_id, desc_data):
    """
    Knowledge Catalog에서 생성된 한글 설명 메타데이터를 BigQuery 데이터셋 설명으로 복사 동기화하고,
    스캔 결과를 연동하기 위한 공식 시스템 라벨을 데이터셋에 부착합니다.
    """
    if not desc_data:
        return
        
    dataset_ref = bq_client.dataset(dataset_id)
    dataset = bq_client.get_dataset(dataset_ref)
    
    # 1. 데이터셋 설명 업데이트
    dataset.description = desc_data.get("description", dataset.description)
    
    # 2. 공식 데이터 설명 퍼블리싱 라벨 추가 (데이터셋 라벨)
    labels = dict(dataset.labels or {})
    scan_id = f"ds-thelook-ds-{dataset_id}".lower().replace("_", "-")
    labels["dataplex-data-documentation-published-scan"] = scan_id
    labels["dataplex-data-documentation-published-project"] = PROJECT_ID
    labels["dataplex-data-documentation-published-location"] = LOCATION
    dataset.labels = labels
    
    # 3. BigQuery에 설명 및 라벨 업데이트 전송
    bq_client.update_dataset(dataset, ["description", "labels"])
    print(f"  [SUCCESS] {dataset_id} 데이터셋 설명 주입 완료!")

In [ ]:
# 메인 실행 파이프라인
print(f"=== 데이터셋 설명 자동화 시작: {DATASET_ID} ===")

# 0. 언어 지침 주입 (Language Directive)
# Gemini 모델이 데이터셋 설명을 한국어로 작성하도록 데이터셋의 기존 설명에 지침을 미리 삽입합니다.
dataset_ref = bq_client.dataset(DATASET_ID)
dataset = bq_client.get_dataset(dataset_ref)
dataset.description = "Generate dataset descriptions using the Korean language"
bq_client.update_dataset(dataset, ["description"])
print(f"  -> {DATASET_ID} 데이터셋에 한국어 작성 지침 주입 완료.")

# 1. Dataset DataScan 생성 또는 로드
scan_id = get_or_create_dataset_datascan(DATASET_ID)

# 2. DataScan 실행 및 완료 대기
run_datascan_and_wait(scan_id)

# 3. 생성된 설명 메타데이터 조회
desc_data = fetch_dataset_generated_description(DATASET_ID)

if desc_data:
    print(f"\n[생성된 데이터셋 설명 요약]")
    print(f" - 설명: {desc_data.get('description')}")
    
    # 4. BigQuery 데이터셋에 설명 및 라벨 반영
    apply_and_publish_dataset_description(DATASET_ID, desc_data)
else:
    print("설명 데이터를 가져오지 못했습니다.")

print(f"\n=== 데이터셋 설명 자동화 완료: {DATASET_ID} ===")